# Replicating network analyses done by Baker et al 2022 on updated FlyWire Connectome data

In [3]:
%matplotlib inline
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import stats
import sys

### Compiling data

In [4]:
connections = pd.read_csv('connections_princeton.csv.gz', compression='gzip')
connections

,pre_root_id,post_root_id,neuropil,syn_count,nt_type
0,720575940625363947,720575940623224444,ME_L,12,GABA
1,720575940630432382,720575940618518557,ME_L,67,ACH
2,720575940627314521,720575940626337738,ME_L,10,GABA
3,720575940620280405,720575940620204726,ME_L,15,ACH
4,720575940636942447,720575940613789411,LA_L,1,GLUT
...,...,...,...,...,...
5342441,720575940610739918,720575940617129885,PLP_L,5,ACH
5342442,720575940653332385,720575940623905719,CRE_R,2,ACH
5342443,720575940622485887,720575940631031803,ME_R,9,ACH
5342444,720575940620020534,720575940636489140,SMP_L,2,ACH


In [5]:
#group by nts so no repeated neuron connections appear

grouped_bynt = connections.groupby(['pre_root_id', 'post_root_id', 'nt_type'], as_index=False)[['syn_count']].sum()
grouped_bynt

,pre_root_id,post_root_id,nt_type,syn_count
0,720575940596125868,720575940605825666,ACH,6
1,720575940596125868,720575940608552405,ACH,6
2,720575940596125868,720575940609975854,ACH,5
3,720575940596125868,720575940613059993,ACH,9
4,720575940596125868,720575940613599129,ACH,6
...,...,...,...,...
3732455,720575940661339777,720575940622727152,ACH,37
3732456,720575940661339777,720575940625109995,ACH,5
3732457,720575940661339777,720575940627311491,ACH,40
3732458,720575940661339777,720575940628084873,ACH,19


In [13]:
#upload classifications csv from flywire

classification = pd.read_csv('classification.csv.gz', compression='gzip')

#filter for auditory system
auditory = (classification[classification['sub_class'] == 'auditory'])
auditory

#currently 393 neurons officially labeled auditory in FlyWire dataset

,root_id,flow,super_class,class,sub_class,hemilineage,side,nerve
23,720575940600433181,afferent,sensory,mechanosensory,auditory,NaN,left,AN
455,720575940604054188,afferent,sensory,mechanosensory,auditory,NaN,left,AN
569,720575940604274622,afferent,sensory,mechanosensory,auditory,NaN,left,AN
726,720575940604531229,afferent,sensory,mechanosensory,auditory,NaN,left,AN
957,720575940604805036,afferent,sensory,mechanosensory,auditory,NaN,left,AN
...,...,...,...,...,...,...,...,...
134482,720575940645086999,afferent,sensory,mechanosensory,auditory,NaN,left,AN
134483,720575940645087511,afferent,sensory,mechanosensory,auditory,NaN,left,AN
134486,720575940645089559,afferent,sensory,mechanosensory,auditory,NaN,left,AN
134969,720575940645539694,afferent,sensory,mechanosensory,auditory,NaN,right,AN


In [7]:
train_dat = pd.read_csv('train_dat.csv')
train_dat

#compile into list of neuron ids
train_dat_ids = train_dat[['Flywire 2024 ID']]
train_dat_ids

,Flywire 2024 ID
0,720575940619783518
1,720575940620062633
2,720575940602914988
3,720575940620195719
4,720575940634508174
...,...
398,720575940655398817
399,720575940630203589
400,720575940617641077
401,720575940612801022


In [8]:
#new baker_neurons csv from paper's supplemental info section
baker_neurons = pd.read_csv('baker_neurons.csv')
baker_neurons

#also compile into list of ids
baker_neurons_ids = baker_neurons[['Root ID (synaptic connectivity)']]
baker_neurons_ids

,Root ID (synaptic connectivity)
0,720575940621039145
1,720575940637390424
2,720575940618522433
3,720575940613535430
4,720575940621301738
...,...
563,720575940600433181
564,720575940630902333
565,720575940610162149
566,720575940616491387


In [ ]:
#find discrepancy between two dataframes

Index([  8,   9,  10,  11,  12,  13,  14,  17,  18,  19,
       ...
       353, 359, 361, 366, 367, 390, 394, 403, 413, 418],
      dtype='int64', length=133)

What is the difference between train_dat and baker_neurons...?

In [10]:
#cross reference baker data with cell types data

celltypes = pd.read_csv('consolidated_cell_types.csv.gz', compression='gzip')
celltypes

,root_id,primary_type,additional_type(s)
0,720575940596125868,T5c,NaN
1,720575940597856265,Tm16,NaN
2,720575940597944841,Tm7,CB3851
3,720575940598267657,TmY15,NaN
4,720575940599333574,Tm1,NaN
...,...,...,...
137672,720575940661335681,CB0180,NaN
137673,720575940661336193,TmY3,NaN
137674,720575940661337217,KCab-p,NaN
137675,720575940661338497,CB0904,NaN


In [11]:
celltypes[celltypes['root_id']==720575940600433181]

#i tested a bunch of root ids from the initial preview of baker_neurons, some neuron ids do not correspond to any neuron in celltypes csv from FlyWire...

,root_id,primary_type,additional_type(s)
23,720575940600433181,JO-B,NaN


In [12]:
celltypes[celltypes['root_id']==720575940619783518]

,root_id,primary_type,additional_type(s)
40142,720575940619783518,CB2364,AVLP_pr01


Since I do not know what is going on with the baker_neurons and train_dat_.csv I am going to run the test on officially labeled auditory neurons from FlyWire classifications df

In [23]:
#collect auditory ids

auditory_flywire_ids = auditory['root_id'].tolist()

#filter in connections df

aud_presynaptic = connections[connections['pre_root_id'].isin(auditory_flywire_ids)]
aud_postsynaptic = connections[connections['post_root_id'].isin(auditory_flywire_ids)]

display(aud_presynaptic, aud_postsynaptic)

,pre_root_id,post_root_id,neuropil,syn_count,nt_type
169343,720575940621791664,720575940626818085,AVLP_R,6,ACH
169655,720575940622295585,720575940628412732,SAD,1,ACH
169763,720575940623474871,720575940635282910,AMMC_L,7,ACH
170095,720575940606850761,720575940623084170,AMMC_R,2,ACH
170218,720575940627046010,720575940629443626,AMMC_L,22,ACH
...,...,...,...,...,...
5341832,720575940642163341,720575940621586473,AMMC_L,8,ACH
5341917,720575940640713680,720575940624101524,AMMC_R,1,ACH
5341969,720575940629450366,720575940621889799,SAD,9,ACH
5342264,720575940614332899,720575940644968302,SAD,2,ACH


,pre_root_id,post_root_id,neuropil,syn_count,nt_type
169451,720575940611335859,720575940620622785,AMMC_L,7,GABA
171146,720575940626745387,720575940637389534,SAD,1,GABA
175208,720575940613118826,720575940614805983,AMMC_L,5,GABA
176784,720575940633714402,720575940620661931,AMMC_R,15,GABA
176968,720575940644488727,720575940612615833,AMMC_R,2,GABA
...,...,...,...,...,...
5336728,720575940629672642,720575940613924271,SAD,3,GABA
5337402,720575940608370012,720575940643903904,SAD,6,GABA
5337454,720575940613708812,720575940630630225,AMMC_L,12,GABA
5337889,720575940624005834,720575940640843316,SAD,5,GABA
